In [1]:
# !pip install fuzzywuzzy
!/opt/conda/envs/rapids/bin/python -m pip install -U fuzzywuzzy

In [2]:
# Uncomment line below to install exlib
# !pip install diskcache
import sys; 

ROOT_DIR = '../..'
sys.path.append(f'{ROOT_DIR}/src')



import openai
import os
import json

def load_api_keys(root_dir):
    import json
    with open(f"{root_dir}/API_KEYS2.json", "r") as file:
        api_keys = json.load(file)
    os.environ['OPENAI_API_KEY'] = api_keys['OPENAI_API_KEY']
    os.environ['ANTHROPIC_API_KEY'] = api_keys['ANTHROPIC_API_KEY']
    # os.environ['GOOGLE_API_KEY'] = api_keys['GOOGLE_API_KEY']
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.path.join(root_dir, api_keys['GOOGLE_APPLICATION_CREDENTIALS'])
    os.environ['CACHE_DIR'] = os.path.join(root_dir, 'cache_dir3')
    return api_keys

load_api_keys(ROOT_DIR);

In [3]:
print(os.environ['GOOGLE_APPLICATION_CREDENTIALS'])

../../application_default_credentials.json


# Emotion

In [4]:
import importlib
import sys; sys.path.append("../src")
import emotion
importlib.reload(emotion)
from emotion import EmotionExample, get_llm_generated_answer, isolate_individual_features
from emotion import distill_relevant_features, calculate_expert_alignment_score
from emotion import load_emotion_data, run_pipeline, group_claims_by_category, make_alignment_matrix, categories_list #aggregate_alignment_scores
from llms import load_model
# from cholec import get_llm_generated_answer
# from cholec import CholecExample, CholecDataset, load_model, items_to_examples
# from cholec import isolate_individual_features, distill_relevant_features, calculate_expert_alignment_scores
print(os.environ['GOOGLE_APPLICATION_CREDENTIALS'])

/opt/conda/envs/rapids/lib/python3.10/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


../../application_default_credentials.json


In [5]:
emotion_data =  load_emotion_data()

emotion_labels = {
    0: "admiration",
    1: "amusement",
    2: "anger",
    3: "annoyance",
    4: "approval",
    5: "caring",
    6: "confusion",
    7: "curiosity",
    8: "desire",
    9: "disappointment",
    10: "disapproval",
    11: "disgust",
    12: "embarrassment",
    13: "excitement",
    14: "fear",
    15: "gratitude",
    16: "grief",
    17: "joy",
    18: "love",
    19: "nervousness",
    20: "optimism",
    21: "pride",
    22: "realization",
    23: "relief",
    24: "remorse",
    25: "sadness",
    26: "surprise",
    27: "neutral"
}

emotion_data

,text,labels,id
0,"aha American Sniper, movie genuinely moved me.",[0],ef1ff2k
1,The most intimidating man in football,[0],efgcr7t
2,Ah the good old Russian Right Hook.,[0],efefbw6
3,"This game is so good, nearly every dc characte...",[0],ed2ug0m
4,"Lol, I don't know the game that well",[1],edje46l
...,...,...,...
107,I'm surprised they've gone this long not knowi...,[26],eczkjtx
108,Bernie Sanders and a hairbrush.,[27],efh2vet
109,Alright we have worn them down enough guys.,[27],eekobe6
110,"He was hooking you up, hoping you both could h...",[27],ed3y67j


In [6]:
from tqdm.auto import tqdm
import json

In [7]:
# model = 'gpt-4o'
models = [
    "gpt-5.2-pro-2025-12-11",
    "gpt-5-mini-2025-08-07",
    "claude-opus-4-5-20251101",
    "claude-haiku-4-5-20251001",
    "gemini-2.5-pro",
    "gemini-2.5-flash"
]

eval_model_name = 'gemini-2.5-flash-lite'
eval_model = load_model(eval_model_name)



/home/runai-home/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/runai-home/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.10) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/home/runai-home/.local/lib/python3.10/site-packages/google/api_core/_py

In [8]:
methods = [
    'vanilla', 
    # 'cot', 
    # 'socratic', 
    # 'subq'
]

In [9]:
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [10]:
import json
import copy
from tqdm.auto import tqdm

# for model in models:
#     print(f"=== Using model {model} ===")
#     for method in methods:
        # print(f"=== Using method {method} ===")

model = models[0]
method = methods[0]
    
load_path = os.path.join(ROOT_DIR, f'results/{method}/emotion_{model}.json')
save_path = os.path.join(ROOT_DIR, f'results/{method}/emotion_{model}_{eval_model_name}.json')

with open(load_path) as input_file:
    results = json.load(input_file)
    
results[0].keys()

dict_keys(['text', 'ground_truth', 'llm_label', 'llm_explanation', 'accuracy', 'claims', 'relevant_claims', 'claims_by_category', 'category_alignment_scores', 'alignment_matrix', 'final_aligned_score'])

In [11]:
emotion_data.iloc[0].to_dict()

{'text': 'aha American Sniper, movie genuinely moved me.',
 'labels': [0],
 'id': 'ef1ff2k'}

In [13]:
import json
import copy
from tqdm.auto import tqdm

for model in models:
    print(f"=== Using model {model} ===")
    for method in methods:
        print(f"=== Using method {method} ===")

        load_path = os.path.join(ROOT_DIR, f'results/{method}/emotion_{model}.json')
        save_path = os.path.join(ROOT_DIR, f'results/{method}/emotion_{model}_{eval_model_name}.json')

        with open(load_path) as input_file:
            results = json.load(input_file)

        new_results = []

        num_examples = len(results)
        print("num_examples", num_examples)
        for di in tqdm(range(num_examples)):
            result = results[di]
            
            row = emotion_data.iloc[di].to_dict()
            
            # image = test_dataset[id2idx_mapping[result['id']]]['image']
            text = row['text']
            
            example = EmotionExample(
                text=row['text'],
                ground_truth=emotion_labels[row['labels'][0]],
                llm_label=result['llm_label'],
                llm_explanation=result['llm_explanation']
            )
            
            print("os.environ['GOOGLE_APPLICATION_CREDENTIALS']", os.environ['GOOGLE_APPLICATION_CREDENTIALS'])
            # isolate individual features
            claims = isolate_individual_features(example.llm_explanation, model=eval_model)
            if claims is None:
                continue
            example.claims = [claim.strip() for claim in claims]

            # distill relevant features
            relevant_claims = distill_relevant_features(
                example,
                model=eval_model
            )
            example.relevant_claims = relevant_claims
            
            print("----- Grouping claims by category -----")
            # for example in tqdm(examples):
            # print(example.text)
            claims_by_category = group_claims_by_category(example.relevant_claims, model=eval_model)
            example.claims_by_category = claims_by_category

            print("----- Calculating expert alignment scores -----")
            # for example in tqdm(examples):
            category_alignment_scores = calculate_expert_alignment_score(example.claims_by_category, model=eval_model)
            example.category_alignment_scores = category_alignment_scores
            example.alignment_matrix = make_alignment_matrix(categories_list, example.claims, example.claims_by_category, example.category_alignment_scores)
            final_aligned_score = example.alignment_matrix.max(axis=-1).mean()
            example.final_aligned_score = final_aligned_score

            # # calculate expert alignment scores
            # # for claim in example.relevant_claims:
            # alignment_scores = []
            # alignment_categories = []
            # alignment_reasonings = []
            # for claim in example.relevant_claims:
            #     category, alignment_score, reasoning = calculate_expert_alignment_score(claim, model=eval_model)
            #     if category is None:
            #         continue
            #     alignment_scores.append(alignment_score)
            #     alignment_categories.append(category)
            #     alignment_reasonings.append(reasoning)
            # example.alignment_scores = alignment_scores
            # example.alignment_categories = alignment_categories
            # example.alignment_reasonings = alignment_reasonings
            # example.final_alignment_score = aggregate_alignment_scores(alignment_scores, len(example.claims))
                
            # align_infos = calculate_expert_alignment_scores(
            #     example.relevant_claims, 
            #     eval_model,
            # )

#             alignable_claims = [info["Claim"] for info in align_infos]
#             alignment_categories = [info["Category"] for info in align_infos]
#             aligned_category_ids = [info["Category ID"] for info in align_infos]
#             alignment_scores = [info["Alignment"] for info in align_infos]
#             alignment_raws = [info["Alignment Raw"] for info in align_infos]
#             alignment_reasonings = [info["Reasoning"] for info in align_infos]
            
#             example.alignable_claims = alignable_claims
#             example.alignment_categories = alignment_categories
#             example.aligned_category_ids = aligned_category_ids
#             example.alignment_scores = alignment_scores
#             example.alignment_raws = alignment_raws
#             example.alignment_reasonings = alignment_reasonings
            
            # Non-alignable claims are given a score of 0.0
            # if len(align_infos) > 0:
            #     example.final_alignment_score = sum(example.alignment_scores) / len(example.claims)
            # else:
            #     example.final_alignment_score = 0.0
            
            # save
            save_dict = {}
            for k, v in example.__dict__.items():
                save_dict[k] = v if not isinstance(v, torch.Tensor) else v.cpu().numpy().tolist()
            # with open(save_path, 'wt') as output_file:
            #     json.dump(save_dict, output_file)

            new_results.append(save_dict)


        with open(save_path, 'wt') as output_file:
            json.dump(new_results, output_file, indent=4)

=== Using model gpt-5.2-pro-2025-12-11 ===
=== Using method vanilla ===
num_examples 100


  0%|          | 0/100 [00:00<?, ?it/s]

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] ../../application_default_credentials.json



100%|██████████| 12/12 [00:00<00:00, 88.58it/s][A

----- Grouping claims by category -----


...


100%|██████████| 11/11 [00:11<00:00,  1.07s/it]


----- Grouping claims by category -----
----- Calculating expert alignment scores -----
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] ../../application_default_credentials.json



100%|██████████| 15/15 [00:19<00:00,  1.27s/it]


----- Grouping claims by category -----


ValueError: not enough values to unpack (expected 1, got 0)

# Cholec

# Emotion